# Kalman Filter Implementation in Object Tracking
A Kalman filter is an algorithm to estimate a state of an object in a given dynamic system. The estimated state could be in e.g. tracking the object on another frame. At its core the Kalman filter consists of two states the predict and the update function.

In [3]:
import numpy as np

## Predtiction
In the prediction step we make assumptions about the next state of the system by considering the motion model which is present in the given dynamic system. In tracking for example we consider a basic time-to-motion model where each object travels at a certain velocity.
$$
\hat x = Fx+u
$$
What we also would like to predict before consider our measurement are the co-varaince of the next state. The provides the uncertainty about our predictions.

$$
\hat P = FPF^T
$$

| Variable | Description | Example Value |
| -------- | ----------- | ------------- |
| $\hat x$ | predicted state without measurement | A position with velcotiy $\hat x = [102.2, 80.2, 1.2, 2.0]$ |
| $x$ | The current state | Could be at the beginning just the position of the bounding box |
| $F$ | The transition function, could be some phyiscal model. | Maybe just time and velcoity $F_k^T = [1 1 0 0]$ | 
| $u$ | The control input, this is mostly in moving system like robots. |  Maybe something like, pressing accelartion $u_k = 0.5$ | 
| $\hat P$ | The covariance matrix representing uncertainty, result of state-transition and co-variance | A covaraince matrix with high values might impress high uncertainty about the expressed prediction. | 
| $P$ | The current co-variance matrix which reflects the uncertainty. | Similar as for $\hat P$ |
| $Q$ | Process noise. | An object might accelerate or change direction |


In [25]:
def predict_state(current_state, transition_object, control_input=None):
    new_state = transition_object @ current_state
    if control_input is not None:
        new_state = new_state + np.asarray(control_input)
    return np.asarray(new_state)

def predict_uncertainty(transition_object, current_uncertainty, process_noise=None):
    predicted_uncertainty = transition_object @ current_uncertainty @ transition_object.T
    if process_noise is not None:
        predicted_uncertainty = predicted_uncertainty + np.asarray(process_noise)
    return predicted_uncertainty

In [26]:
current_state = np.array([102.2, 80.2, 1.2, 2.0])

transition_object = np.array([
              [1, 0, 0.5, 0],
              [0, 1, 0, 0.5],
              [0, 0, 1, 0],
              [0, 0, 0, 1]])

uncertainty = np.array([
              [25, 0, 0, 0],
              [0, 25, 0, 0],
              [0, 0, 100, 0],
              [0, 0, 0, 100]])

# Example process noise (simple diagonal placeholder)
process_noise = np.diag([1, 1, 10, 10])

In [27]:
predict_state(current_state, transition_object)

array([102.8,  81.2,   1.2,   2. ])

In [28]:
predict_uncertainty(transition_object, uncertainty, process_noise)

array([[ 51.,   0.,  50.,   0.],
       [  0.,  51.,   0.,  50.],
       [ 50.,   0., 110.,   0.],
       [  0.,  50.,   0., 110.]])

## Update
Now in the update step we consider the actualy measurements from our sensor or detection system so that we can see how far they diverge from our prediction. First we compute the innovation that provids us with the inital differenc of the measurement.

$$
\hat y = z_k - H\hat x
$$

Further we again combute covariance, but this time considering them as resudial for uncertainty, so only the predicted co-varainces that
also fit a measurement with a given noise from the senosr.

$$
S = H\hat PH^T + R
$$

Now to use the given uncertainty for the measurement we check "how much we trust the measurment in consideration of the difference $\hat y$ and the uncertainty $S$?" with the Kalman-gain.

$$
K_k = P_{k|k-1} H^T S^{-1}_k
$$

Now inutitivly whether we trust our measurement or we trust our prediction depends, once on the noisy and on the otherhand on the uncertainty. Thus with a large $R$ we should not trust the measurement but with large $P$ we should not trust our prediction. This is reflected in the Kalman-gain $K_k$.
Now we update our state and the Co-variance uncertainty for the next frame. Frist we update the state using the difference in measurement
and in consideration with how much percent we consider the prediction as correct. If $K$ is high we trust our prediction more and the refelected differenc in $y$ has a stronger influence in the next state.

$$
\hat x = x + yK
$$

Now also updating the uncertainty in a similar way, whenever $K$ is high our uncertainty should decrease since our prediction was better than the acutal measurement to represent the new state of the object. Thus we compute:

$$
\hat P = (I - HK)P_{k-1}
$$

| Variable | Description | Example Value |
| -------- | ----------- | ------------- |
| $z_k$ | Difference between measurement and prediction. | Large if measurement is mostly different to prediction. |
| $\hat y$  | The innovation value that corresponds to the inital difference of measurement and prediction state | Could be negativ, zero or postive |
| $H$ | This matrix reflect the measurments which are taken, sometimes only postion or only velocity is measured | In Object tracking this could be just the postion of the next frame | 
| $\hat x$ | The prediction from the prediction step | Contains $\hat x = [x_k, y_k, v_k, u_k]$ |
| $R$ | A noise fitting to the given measurement | Could be for example white noise $N(0, \sigma)$ |
| $K_k$ | How trustworthy a measurement or prediciton is | A large $K_k$ is considered as trustworthy prediction |
| $I$ | Identity matrix | A matrix where the diagnal values are ones. |
| $S$ | Uncertainty resudial. | Only the relevant unvcertainty remains. |


In [2]:
def innovation_value(measurements, prediction, measurement_prediction_connection):
    if len(prediction) != len(measurement_prediction_connection):
        raise ValueError("Dimensions need to be equal.")
    connected_prediction = measurement_prediction_connection @ prediction
    return measurements - connected_prediction

def uncertainty_resudial(predicted_uncertainty, measurement_prediction_connection, measurement_noise):
    if len(predicted_uncertainty) != len(measurement_prediction_connection):
        raise ValueError("Dimensions need to be equal")
    connected_predicted_uncertainty = measurement_prediction_connection @ predicted_uncertainty
    return connected_predicted_uncertainty @ measurement_prediction_connection.T + measurement_noise

def kalman_gain(previous_predicted_uncertainty, measurement_prediction_connection, uncertainty_resudial):
    if len(previous_predicted_uncertainty) != len(measurement_prediction_connection) != len(uncertainty_resudial):
        raise ValueError("Dimensions need to be equal")
    return previous_predicted_uncertainty @ measurement_prediction_connection.T @ np.linalg.inv(uncertainty_resudial)

def next_state(current_state, innovation_value, kalman_gain):
    if len(current_state) != len(innovation_value) != len(kalman_gain):
        raise ValueError("Dimensions need to be equal")
    return current_state + kalman_gain @ innovation_value

def next_uncertainty(measurement_prediction_connection, kalman_gain, previous_uncertainty):
    if len(measurement_prediction_connection) != len(kalman_gain) != len(previous_uncertainty):
        raise ValueError("Dimensions need to be equal")
    return (np.identity() - kalman_gain @ measurement_prediction_connection) @ previous_uncertainty